# Day 1: Python Foundations for Modeling and Inference

Advanced Pathway Project 1: Epidemic Modeling and Inference

By the end of today, participants should be able to:

- write small Python functions for mathematical models
- use NumPy arrays to store states, time grids, and observations
- make plots that communicate simulation results
- use loops to simulate repeated updates over time
- represent a Markov chain with a transition matrix
- simulate random events and summarize Monte Carlo results
- connect likelihood, parameters, and noisy observations before Day 3 MCMC

## Day 1 schedule

| Time | Focus |
|---|---|
| 8:00-8:30 | Shared reimbursement meeting and program logistics |
| 8:30-8:50 | Advanced pathway welcome, three-day plan, instructors, participant introductions |
| 8:50-10:15 | Python warm-up: variables, functions, arrays, and plotting |
| 10:15-10:30 | Break |
| 10:30-12:00 | Loops, model updates, and simulation |
| 12:00-1:00 | Lunch |
| 1:00-2:15 | Randomness, probability, and transition matrices |
| 2:15-2:30 | Break |
| 2:30-3:45 | Likelihood, simulation, and Monte Carlo intuition |
| 3:45-4:00 | Reflection and Day 2 preview |

## 1. Python as a Modeling Notebook

Python notebooks let us mix explanation, code, results, and reflection. In this project, we will use Python to describe systems that change over time. The basic pattern will be:

1. choose a model state, such as the number of susceptible and infected people
2. choose parameters, such as infection and recovery rates
3. write an update rule
4. repeat the update rule
5. visualize and interpret the result

In [ ]:
# Core libraries for Days 1 and 2
import numpy as np
import matplotlib.pyplot as plt

# Keep plots readable in the notebook
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

## 2. Variables, Expressions, and Types

A variable stores a value so we can reuse it. In modeling, variables often represent parameters, initial conditions, or time steps.

In [ ]:
population = 1000
initial_infected = 5
initial_susceptible = population - initial_infected
initial_recovered = 0

beta = 0.30    # infection rate
gamma = 0.10   # recovery rate
dt = 1.0       # one day per model step

initial_susceptible, initial_infected, initial_recovered

### Exercise 1: Read and Modify Parameters

1. Change `initial_infected` from 5 to 25.
2. Recompute the susceptible group.
3. Predict whether infections should rise faster or slower.
4. Run the cell and compare your prediction with a neighbor.

In [ ]:
# Exercise 1 workspace
population = 1000
initial_infected = 5
initial_susceptible = population - initial_infected
initial_recovered = 0

print("S0 =", initial_susceptible)
print("I0 =", initial_infected)
print("R0 =", initial_recovered)

## 3. Functions for Model Rules

A function lets us name a reusable calculation. Day 2 uses functions for the right-hand side of a differential equation. Day 3 uses functions for likelihoods and model simulators.

In [ ]:
def new_infections(S, I, beta, population):
    """Expected new infections in one time step."""
    return beta * S * I / population

new_infections(995, 5, beta=0.30, population=1000)

In [ ]:
def sir_one_step(S, I, R, beta, gamma, population, dt=1.0):
    """Advance a simple SIR model by one Euler step."""
    infections = beta * S * I / population
    recoveries = gamma * I

    S_next = S - dt * infections
    I_next = I + dt * infections - dt * recoveries
    R_next = R + dt * recoveries

    return S_next, I_next, R_next

sir_one_step(995, 5, 0, beta=0.30, gamma=0.10, population=1000)

### Exercise 2: Function Checkpoint

The SIR step has two flows: susceptible people move into infected, and infected people move into recovered.

1. Increase `gamma` to 0.25. What changes?
2. Decrease `beta` to 0.10. What changes?
3. Check whether `S + I + R` stays equal to the total population.

In [ ]:
# Exercise 2 workspace
S1, I1, R1 = sir_one_step(995, 5, 0, beta=0.30, gamma=0.10, population=1000)
print(S1, I1, R1)
print("Total:", S1 + I1 + R1)

## 4. Lists and NumPy Arrays

A list stores multiple values. A NumPy array is better for numerical work because it supports vectorized mathematical operations and matrix multiplication.

In [ ]:
python_list = [995, 5, 0]
state = np.array([995.0, 5.0, 0.0])

print("List:", python_list)
print("Array:", state)
print("Array times 2:", state * 2)

In [ ]:
time = np.arange(0, 11, 1)
infected = np.array([5, 6, 7, 9, 12, 16, 20, 24, 28, 31, 33])

plt.plot(time, infected, marker="o")
plt.xlabel("Day")
plt.ylabel("Number infected")
plt.title("Example infection counts")
plt.show()

### Exercise 3: Plotting Checkpoint

Use the workspace below to create a new plot.

1. Create a NumPy array called `recovered` with values that increase over time.
2. Plot both `infected` and `recovered` on the same axes.
3. Add a legend.
4. Write one sentence interpreting the plot.

In [ ]:
# Exercise 3 workspace
time = np.arange(0, 11, 1)
infected = np.array([5, 6, 7, 9, 12, 16, 20, 24, 28, 31, 33])

# Add recovered values here
recovered = np.array([0, 1, 1, 2, 3, 5, 7, 10, 14, 18, 23])

plt.plot(time, infected, marker="o", label="Infected")
plt.plot(time, recovered, marker="s", label="Recovered")
plt.xlabel("Day")
plt.ylabel("People")
plt.title("Two epidemic time series")
plt.legend()
plt.show()

## 5. Loops for Simulation

A simulation repeats a rule many times. The current state becomes the starting point for the next step. This is the core idea behind Euler's method in Day 2.

In [ ]:
def simulate_sir(days, S0, I0, R0, beta, gamma, population, dt=1.0):
    steps = int(days / dt)
    t = np.arange(steps + 1) * dt
    states = np.zeros((steps + 1, 3))
    states[0] = np.array([S0, I0, R0])

    for k in range(steps):
        S, I, R = states[k]
        states[k + 1] = sir_one_step(S, I, R, beta, gamma, population, dt)

    return t, states

t, states = simulate_sir(
    days=60,
    S0=995,
    I0=5,
    R0=0,
    beta=0.30,
    gamma=0.10,
    population=1000,
)

states[:5]

In [ ]:
plt.plot(t, states[:, 0], label="Susceptible")
plt.plot(t, states[:, 1], label="Infected")
plt.plot(t, states[:, 2], label="Recovered")
plt.xlabel("Day")
plt.ylabel("People")
plt.title("SIR simulation")
plt.legend()
plt.show()

### Exercise 4: Simulation Lab

This is a natural mid-morning break point.

1. Run the simulation with `beta = 0.20`, `0.30`, and `0.45`.
2. For each value, record the peak number infected.
3. Which parameter value produces the largest peak?
4. How might a classroom connect this to public health decisions?

In [ ]:
# Exercise 4 workspace
for beta_value in [0.20, 0.30, 0.45]:
    t, states = simulate_sir(60, 995, 5, 0, beta_value, 0.10, 1000)
    peak_infected = states[:, 1].max()
    peak_day = t[states[:, 1].argmax()]
    print(f"beta={beta_value:.2f}: peak infected={peak_infected:.1f} on day {peak_day:.0f}")

## 6. Randomness and Reproducibility

Day 2 includes Markov chains, and Day 3 includes MCMC. Both depend on random choices. A random seed makes results reproducible.

In [ ]:
rng = np.random.default_rng(seed=42)

coin_flips = rng.choice(["H", "T"], size=10, p=[0.5, 0.5])
coin_flips

In [ ]:
# Simulate whether each infected person recovers today.
rng = np.random.default_rng(seed=42)
infected_people = 20
recovery_probability = 0.10
recoveries = rng.binomial(n=infected_people, p=recovery_probability)
recoveries

### Exercise 5: Random Simulation Checkpoint

1. Change `recovery_probability` to 0.25.
2. Run the simulation several times with different seeds.
3. Why is the result not always the same?
4. What does this suggest about interpreting one simulation run?

In [ ]:
# Exercise 5 workspace
for seed in [1, 2, 3, 4, 5]:
    rng = np.random.default_rng(seed=seed)
    recoveries = rng.binomial(n=20, p=0.10)
    print(seed, recoveries)

## 7. Transition Matrices for Markov Chains

A Markov chain describes transitions between states. The next state depends on the current state and a transition rule. Day 2 uses this idea more formally.

Suppose a person can be in one of three states: Susceptible, Infected, or Recovered. The transition matrix below stores the one-day probabilities of moving between states.

In [ ]:
# Rows are current states, columns are next states.
# State order: S, I, R
T = np.array([
    [0.92, 0.08, 0.00],
    [0.00, 0.85, 0.15],
    [0.00, 0.00, 1.00],
])

initial_distribution = np.array([0.99, 0.01, 0.00])
next_distribution = initial_distribution @ T
next_distribution

In [ ]:
distribution = initial_distribution.copy()
history = [distribution]

for day in range(30):
    distribution = distribution @ T
    history.append(distribution)

history = np.array(history)

plt.plot(history[:, 0], label="S")
plt.plot(history[:, 1], label="I")
plt.plot(history[:, 2], label="R")
plt.xlabel("Day")
plt.ylabel("Probability")
plt.title("Markov chain distribution over time")
plt.legend()
plt.show()

### Exercise 6: Transition Matrix Break

1. Check that each row of `T` adds to 1.
2. Increase the probability of moving from susceptible to infected.
3. Decrease the probability of staying infected.
4. Replot the distributions and describe the change.

In [ ]:
# Exercise 6 workspace
print("Row sums:", T.sum(axis=1))

T_new = np.array([
    [0.90, 0.10, 0.00],
    [0.00, 0.75, 0.25],
    [0.00, 0.00, 1.00],
])

print("New row sums:", T_new.sum(axis=1))

## 8. From Model Output to Data

Real data are noisy. Day 3 asks: if the model has unknown parameters, which parameter values make the observed data most plausible?

We will start with a simulated infected curve and add observation noise.

In [ ]:
t, true_states = simulate_sir(40, 995, 5, 0, beta=0.30, gamma=0.10, population=1000)
true_infected = true_states[:, 1]

rng = np.random.default_rng(seed=123)
observed_infected = rng.poisson(lam=np.maximum(true_infected, 0.1))

plt.plot(t, true_infected, label="Model infected")
plt.scatter(t, observed_infected, s=20, color="black", label="Noisy observations")
plt.xlabel("Day")
plt.ylabel("Infected people")
plt.title("Model output and noisy observations")
plt.legend()
plt.show()

## 9. A First Likelihood Score

A likelihood compares observed data with model predictions. A simple score is the sum of squared errors. Smaller values mean the model curve is closer to the observed data.

This is not the full Bayesian likelihood used in Day 3, but it gives the right intuition: parameters are judged by how well their simulated output explains the data.

In [ ]:
def simulate_infected_curve(beta, gamma=0.10):
    t, states = simulate_sir(40, 995, 5, 0, beta=beta, gamma=gamma, population=1000)
    return states[:, 1]

def squared_error_score(beta, observed):
    predicted = simulate_infected_curve(beta)
    return np.sum((observed - predicted) ** 2)

for beta_value in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]:
    print(beta_value, squared_error_score(beta_value, observed_infected))

In [ ]:
beta_grid = np.linspace(0.10, 0.50, 81)
scores = np.array([squared_error_score(beta_value, observed_infected) for beta_value in beta_grid])
best_beta = beta_grid[np.argmin(scores)]

plt.plot(beta_grid, scores)
plt.axvline(best_beta, color="black", linestyle="--", label=f"best beta = {best_beta:.3f}")
plt.xlabel("beta")
plt.ylabel("squared error score")
plt.title("Parameter search by simulation")
plt.legend()
plt.show()

best_beta

### Exercise 7: Inference Preview

This is a natural afternoon break point.

1. Change the true data-generating value from `beta = 0.30` to another value.
2. Regenerate the noisy observations.
3. Run the grid search again.
4. Does the best beta recover the true beta exactly? Why or why not?

In [ ]:
# Exercise 7 workspace
true_beta = 0.30
rng = np.random.default_rng(seed=123)
_, true_states = simulate_sir(40, 995, 5, 0, beta=true_beta, gamma=0.10, population=1000)
observed = rng.poisson(lam=np.maximum(true_states[:, 1], 0.1))

beta_grid = np.linspace(0.10, 0.50, 81)
scores = np.array([squared_error_score(beta_value, observed) for beta_value in beta_grid])
print("True beta:", true_beta)
print("Best beta:", beta_grid[np.argmin(scores)])

## 10. Monte Carlo Thinking

Monte Carlo methods repeat random experiments many times. MCMC, used on Day 3, is a special Monte Carlo method that builds a Markov chain whose long-run behavior helps approximate a probability distribution.

Before MCMC, we can practice with a simpler question: if the same epidemic model produces noisy data many times, how much does our best parameter estimate vary?

In [ ]:
def estimate_beta_from_noisy_data(seed, true_beta=0.30):
    rng = np.random.default_rng(seed=seed)
    _, true_states = simulate_sir(40, 995, 5, 0, beta=true_beta, gamma=0.10, population=1000)
    observed = rng.poisson(lam=np.maximum(true_states[:, 1], 0.1))
    beta_grid = np.linspace(0.10, 0.50, 81)
    scores = np.array([squared_error_score(beta_value, observed) for beta_value in beta_grid])
    return beta_grid[np.argmin(scores)]

estimates = np.array([estimate_beta_from_noisy_data(seed) for seed in range(50)])

plt.hist(estimates, bins=12, edgecolor="white")
plt.axvline(0.30, color="black", linestyle="--", label="true beta")
plt.xlabel("Estimated beta")
plt.ylabel("Number of simulations")
plt.title("Monte Carlo variation in beta estimates")
plt.legend()
plt.show()

print("Mean estimate:", estimates.mean())
print("Standard deviation:", estimates.std())

### Exercise 8: End-of-Day Synthesis

Work with a partner and answer these questions.

1. Which Python idea from today appears in the SIR notebook?
2. Which Python idea from today appears in the Markov chains notebook?
3. Which Python idea from today appears in the MCMC notebook?
4. What is one classroom question you could ask using simulation?
5. What part of Day 2 or Day 3 feels most important to review tomorrow?

## Day 2 Preview

Tomorrow we will use the same building blocks in a more formal way:

- arrays store model states
- functions define model dynamics
- loops and numerical solvers advance the model
- plots communicate how a system changes over time
- transition matrices describe stochastic movement between states

Keep this notebook as a reference while working through the SIR, Euler method, and Markov chain materials.